In [1]:
# soybean_weekly_prices_fixed.py
# Wymagane: pip install yfinance pandas
# Uruchom: python soybean_weekly_prices_fixed.py

import os
import pandas as pd
import yfinance as yf

# === USTAWIENIA ===
OUTPUT_PATH = r"D:\marcinszwagrzyk.github.io\Soy\data\soybean_weekly_prices.csv"
START_DATE = "2010-01-01"
TICKERS = {
    "Soybeans": "ZS=F",   # Soybean futures (CBOT)
    "ADM": "ADM",         # Archer Daniels Midland
    "Bunge": "BG"         # Bunge Limited (NYSE ticker BG)
}

# === POMOCNICZE ===
def choose_price_column(df):
    """
    Zwraca Series z ceną tygodniową, wybierając najlepszą kolumnę.
    Priorytet: 'Adj Close' -> 'Adj_Close' -> 'Close' -> 'Last'
    """
    cols = list(df.columns)
    # debug: pokaż kolumny
    print("   Kolumny w DataFrame:", cols)
    if "Adj Close" in df.columns:
        return df["Adj Close"]
    if "Adj_Close" in df.columns:
        return df["Adj_Close"]
    if "Close" in df.columns:
        return df["Close"]
    if "Last" in df.columns:
        return df["Last"]
    # brak rozpoznanej kolumny
    return None

def download_weekly(ticker_symbol, start_date):
    """
    Pobiera tygodniowe dane z yfinance i zwraca DataFrame z kolumnami Date + cena.
    """
    print(f"Pobieram {ticker_symbol} od {start_date} (interval=1wk)...")
    try:
        df = yf.download(ticker_symbol, start=start_date, interval="1wk", progress=False, auto_adjust=False)
    except Exception as e:
        print(f"  Błąd podczas pobierania {ticker_symbol}: {e}")
        return None

    if df is None or df.empty:
        print(f"  ⚠ Brak danych dla {ticker_symbol}")
        return None

    # Niektóre zwroty mają MultiIndex lub nietypowe nazwy, więc debugujemy i wybieramy kolumnę
    price_series = choose_price_column(df)
    if price_series is None:
        print(f"  ⚠ Nie znaleziono kolumny cenowej dla {ticker_symbol}; pomijam.")
        return None

    # Przygotuj DF: Date + price (nazwij kolumnę docelowo poza funkcją)
    out = price_series.reset_index()
    # ujednolicenie nazwy kolumny index->Date
    out.rename(columns={out.columns[0]: "Date", out.columns[1]: "Price"}, inplace=True)
    out["Date"] = pd.to_datetime(out["Date"])
    out = out.sort_values("Date").reset_index(drop=True)
    return out

# === POBIERANIE I SCALANIE ===
frames = []
for nice_name, ticker in TICKERS.items():
    df_price = download_weekly(ticker, START_DATE)
    if df_price is None:
        continue
    # przemianuj kolumnę Price na przyjazną nazwę
    df_price = df_price.rename(columns={"Price": nice_name})
    frames.append(df_price)

if not frames:
    raise RuntimeError("Nie pobrano żadnych danych. Sprawdź połączenie/internet i dostępność tickerów.")

# Scalaj po Date (outer aby nic nie stracić)
merged = frames[0]
for df in frames[1:]:
    merged = pd.merge(merged, df, on="Date", how="outer")

merged = merged.sort_values("Date").reset_index(drop=True)

# Opcjonalne: zaokrąglenie cen i zapis
merged = merged.copy()
# zostawiamy oryginalne wartości, ale można zaokrąglić:
# merged = merged.round(4)

# Upewnij się, że folder istnieje i zapisz
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
merged.to_csv(OUTPUT_PATH, index=False, float_format="%.4f", date_format="%Y-%m-%d")

print("\n✅ Gotowe!")
print(f"Zapisano plik: {OUTPUT_PATH}")
print(f"Zakres dat: {merged['Date'].min().date()} → {merged['Date'].max().date()}")
print("\nPodgląd ostatnich wierszy:")
print(merged.tail(8))


Pobieram ZS=F od 2010-01-01 (interval=1wk)...
   Kolumny w DataFrame: [('Adj Close', 'ZS=F'), ('Close', 'ZS=F'), ('High', 'ZS=F'), ('Low', 'ZS=F'), ('Open', 'ZS=F'), ('Volume', 'ZS=F')]
Pobieram ADM od 2010-01-01 (interval=1wk)...
   Kolumny w DataFrame: [('Adj Close', 'ADM'), ('Close', 'ADM'), ('High', 'ADM'), ('Low', 'ADM'), ('Open', 'ADM'), ('Volume', 'ADM')]
Pobieram BG od 2010-01-01 (interval=1wk)...
   Kolumny w DataFrame: [('Adj Close', 'BG'), ('Close', 'BG'), ('High', 'BG'), ('Low', 'BG'), ('Open', 'BG'), ('Volume', 'BG')]

✅ Gotowe!
Zapisano plik: D:\marcinszwagrzyk.github.io\Soy\data\soybean_weekly_prices.csv
Zakres dat: 2010-01-01 → 2025-10-10

Podgląd ostatnich wierszy:
Ticker       Date  Soybeans        ADM      Bunge
816    2025-08-22   1028.25  62.660000  84.629997
817    2025-08-29   1012.00  61.939999  81.129997
818    2025-09-05   1015.25  61.419998  81.940002
819    2025-09-12   1037.50  60.830002  79.900002
820    2025-09-19   1012.25  61.070000  79.389999
821    20